In [4]:
!pip install biopython

In [5]:
import os
import pandas as pd
from Bio.PDB import MMCIFParser
from math import sqrt
import itertools
import json

In [6]:
def calculate_distance(atom1, atom2):
    diff = atom1.coord - atom2.coord
    return sqrt((diff ** 2).sum())

In [7]:
def get_distances(chain_A, chain_B, dock_label):
    distances = []
    for atom_A, atom_B in itertools.product(chain_A.get_atoms(), chain_B.get_atoms()):
        dist = calculate_distance(atom_A, atom_B)
        distances.append({
            "atom1_name": atom_A.get_name(),             # Atom name, e.g., "CG2"
            "atom1_residue": atom_A.get_parent().get_resname(),  # Residue name, e.g., "LEU"
            "atom1_residue_num": atom_A.get_parent().get_id()[1],    # Residue sequence number (e.g., 132)
            "atom1_insertion_code": atom_A.get_parent().get_id()[2],    # Insertion code, if present
            "atom1_chain_id": atom_A.get_parent().get_parent().id,  # Chain ID, e.g., "F1"
            "atom2_name": atom_B.get_name(),             # Atom name, e.g., "CG2"
            "atom2_residue": atom_B.get_parent().get_resname(),  # Residue name, e.g., "LEU"
            "atom2_residue_num": atom_B.get_parent().get_id()[1],    # Residue sequence number (e.g., 132)
            "atom2_insertion_code": atom_B.get_parent().get_id()[2],    # Insertion code, if present
            "atom2_chain_id": atom_B.get_parent().get_parent().id,  # Chain ID, e.g., "F1"
            "distance": dist,
            "dock": dock_label
        })
    return sorted(distances, key=lambda x: x["distance"])  # return top 10

In [17]:
# --- Folder structure ---
base_folder = "boltz_results_narcolepsy/predictions"

# --- MMCIF parser ---
parser = MMCIFParser(QUIET=True)

# --- Collect all distance entries ---
all_distances = []
all_affinities = []
all_confidences = []

# --- Traverse subfolders ---
for subfolder in os.listdir(base_folder):
    subfolder_path = os.path.join(base_folder, subfolder)
    if os.path.isdir(subfolder_path):
        # Find the .cif file in the subfolder
        cif_files = [f for f in os.listdir(subfolder_path) if f.endswith(".cif")]
        if not cif_files:
            print(f"No .cif file in {subfolder_path}")
            continue

        cif_path = os.path.join(subfolder_path, cif_files[0])
        try:
            structure = parser.get_structure(subfolder, cif_path)
            model = structure[0]
            chains = list(model.get_chains())

            if len(chains) < 2:
                print(f"Skipping {subfolder}: fewer than 3 chains found")
                continue

            protein_1 = chains[0]
            drug = chains[1]

            # Compute and collect distances
            all_distances.extend(get_distances(protein_1, drug, subfolder))

        except Exception as e:
            print(f"Error processing {cif_path}: {e}")

        affinity_files = [f for f in os.listdir(subfolder_path) if f.startswith("affinity_")]

        if not affinity_files:
            print(f"No affinity file in {subfolder_path}")
            continue

        affinity_path = os.path.join(subfolder_path, affinity_files[0])

        affinity = None
        with open(affinity_path) as affinity_file:
            affinity = json.load(affinity_file)

        if affinity is not None:
            affinity["dock"] = subfolder
            all_affinities.extend([affinity])

        confidence_files = [f for f in os.listdir(subfolder_path) if f.startswith("confidence_")]

        if not confidence_files:
            print(f"No confidence file in {subfolder_path}")
            continue

        confidence_path = os.path.join(subfolder_path, confidence_files[0])

        confidence = None
        with open(confidence_path) as confidence_file:
            confidence = json.load(confidence_file)

        if confidence is not None:
            confidence["dock"] = subfolder
            all_confidences.extend([confidence])


# --- Convert to DataFrame ---
distance_df = pd.DataFrame(all_distances)
affinity_df = pd.DataFrame(all_affinities)
confidence_df = pd.DataFrame(all_confidences)

In [8]:
distance_df_condensed = distance_df.groupby('dock').apply(lambda g: pd.Series({
    'percent_hit': 100 * (((g['atom1_residue_num'].between(61, 69) | g['atom1_residue_num'].between(79, 115)) & (g['distance'] < 4)).sum() / len(g))
})).reset_index()

result_df = distance_df_condensed.merge(affinity_df, on='dock').merge(confidence_df, on='dock')
result_df[['company', 'patent_ref_number', 'dimer']] = result_df['dock'].str.split('_', expand=True)

/tmp/ipykernel_5882/1780998216.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  distance_df_condensed = distance_df.groupby('dock').apply(lambda g: pd.Series({


In [9]:
distance_df.to_csv('distance_df_narcolepsy.csv')
affinity_df.to_csv('affinity_df_narcolepsy.csv')
confidence_df.to_csv('confidence_df_narcolepsy.csv')

In [7]:
result_df

NameError: name 'result_df' is not defined

In [9]:
affinity_df

,affinity_pred_value,affinity_probability_binary,affinity_pred_value1,affinity_probability_binary1,affinity_pred_value2,affinity_probability_binary2,dock
0,-0.189116,0.359357,-0.650446,0.481031,0.272214,0.237684,tak_994
1,-0.362428,0.128955,-0.759751,0.072066,0.034895,0.185843,tak_861
2,-2.196406,0.954557,-1.802659,0.998619,-2.590152,0.910495,tak_925
3,-0.249006,0.752303,-0.477577,0.891559,-0.020435,0.613047,alks_2680


In [ ]:
Extracellular_Domain
1:54
Helical_1
55:75
Cytoplasmic_1
76:88
Helical_2
89:110
Extracellular_2
111:127
Helical_3
128:150
Cytoplasmic_2
151:170
Helical_4
171:191
Extracellular_3
192:222
Helical_5
223:243
Cytoplasmic_3
244:304
Helical_6
305:326
Extracellular_4
327:342
Helical_7
343:366
Cytoplasmic
367:444

In [10]:
distance_df

,atom1_name,atom1_residue,atom1_residue_num,atom1_insertion_code,atom1_chain_id,atom2_name,atom2_residue,atom2_residue_num,atom2_insertion_code,atom2_chain_id,distance,dock
0,CD2,TYR,481,,A,F30,LIG1,1,,B,2.375836,tak_994
1,CE2,TYR,481,,A,F30,LIG1,1,,B,2.615386,tak_994
2,ND2,ASN,488,,A,F29,LIG1,1,,B,2.856614,tak_994
3,OE1,GLN,194,,A,O32,LIG1,1,,B,2.891116,tak_994
4,CD,GLN,194,,A,O32,LIG1,1,,B,3.135199,tak_994
...,...,...,...,...,...,...,...,...,...,...,...,...
563745,OD1,ASP,291,,A,C58,LIG1,1,,B,87.473439,alks_2680
563746,CG,LYS,323,,A,C58,LIG1,1,,B,87.530081,alks_2680
563747,OD2,ASP,291,,A,C58,LIG1,1,,B,87.734463,alks_2680
563748,NZ,LYS,323,,A,C58,LIG1,1,,B,87.937228,alks_2680


In [1]:
receptor = "DYKDDDDAMGTKLEDSPPCRNWSSASELNETQEPFLNPTDYDDEEFLRYLWREYLHPKEYEWVLIAGYIIVFVVALIGNVLVCVAVWKNHHMRTVTNYFIVNLSLADVLVTITCLPATLVVDITETWFFGQSLCKVIPYLQTVSVSVSVLTLSCIALDRWYAICHPLMFKSTAKRARNSIVIIWIVSCIIMIPQAIVMECSTVFPGLANKTTLFTVCDERWGGEIYPKMYHICFFLVTYMAPLCLMVLAYLQIFRKLWCRQGIDCSFWNESYLTGSRDERKKSLLSKFGMDEGVTFMFIGRFDRGQKGVDVLLKAIEILSSKKEFQEMRFIIIGKGDPELEGWARSLEEKHGNVKVITEMLSREFVRELYGSVDFVIIPSYFEPFGLVALEAMCLGAIPIASAVGGLRDIITNETGILVKAGDPGELANAILKALELSRSDLSKFRENCKKRAMSFSKQIRARRKTARMLMVVLLVFAICYLPISILNVLKRVFGMFAHTEDRETVYAWFTFSHWLVYANSAANPIIYNFLSGKFREEFKAAFSCCCLGVHHHHHHHHHH"
print(len(receptor))

560


In [2]:
uniprot_receptor = "MSGTKLEDSPPCRNWSSASELNETQEPFLNPTDYDDEEFLRYLWREYLHPKEYEWVLIAGYIIVFVVALIGNVLVCVAVWKNHHMRTVTNYFIVNLSLADVLVTITCLPATLVVDITETWFFGQSLCKVIPYLQTVSVSVSVLTLSCIALDRWYAICHPLMFKSTAKRARNSIVIIWIVSCIIMIPQAIVMECSTVFPGLANKTTLFTVCDERWGGEIYPKMYHICFFLVTYMAPLCLMVLAYLQIFRKLWCRQIPGTSSVVQRKWKPLQPVSQPRGPGQPTKSRMSAVAAEIKQIRARRKTARMLMIVLLVFAICYLPISILNVLKRVFGMFAHTEDRETVYAWFTFSHWLVYANSAANPIIYNFLSGKFREEFKAAFSCCCLGVHHRQEDRLTRGRTSTESRKSLTTQISNFDNISKLSEQVVLTSISTLPAANGAGPLQNW"
print(len(uniprot_receptor))

444


In [12]:
df_contacts

,atom1_name,atom1_residue,atom1_residue_num,atom1_insertion_code,atom1_chain_id,atom2_name,atom2_residue,atom2_residue_num,atom2_insertion_code,atom2_chain_id,distance,dock
0,CD2,TYR,481,,A,F30,LIG1,1,,B,2.375836,tak_994
1,CE2,TYR,481,,A,F30,LIG1,1,,B,2.615386,tak_994
2,ND2,ASN,488,,A,F29,LIG1,1,,B,2.856614,tak_994
3,OE1,GLN,194,,A,O32,LIG1,1,,B,2.891116,tak_994
4,CD,GLN,194,,A,O32,LIG1,1,,B,3.135199,tak_994
...,...,...,...,...,...,...,...,...,...,...,...,...
433005,CB,ALA,117,,A,O33,LIG1,1,,B,3.939284,alks_2680
433006,ND1,HIS,514,,A,C49,LIG1,1,,B,3.944008,alks_2680
433007,OE1,GLN,141,,A,C52,LIG1,1,,B,3.959134,alks_2680
433008,CE2,TYR,518,,A,C50,LIG1,1,,B,3.973565,alks_2680


In [11]:
df_contacts = distance_df[distance_df["distance"] < 4]

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def plot_docks_with_segments(distance_df: pd.DataFrame,
                             values=None,
                             values_style: dict | None = None,
                             title: str | None = None):
    """
    distance_df must contain columns: 'dock' and 'Distance'
    - 'dock': categorical label
    - 'Distance': numeric position along x in [1, 444]

    Optional:
    - values: a 1D array-like of length 444 to plot as a line on top (e.g., a score per position)
    - values_style: dict of matplotlib kwargs for that line (e.g., {'linewidth': 1.5})
    """
    # --- Validate inputs ---
    # required_cols = {"dock", "distance"}
    # if not required_cols.issubset(distance_df.columns):
    #     missing = required_cols - set(distance_df.columns)
    #     raise ValueError(f"distance_df is missing required columns: {sorted(missing)}")

    # Normalize positions to ints within [1,444]
    pos = (pd.to_numeric(distance_df["atom1_residue_num"], errors="coerce")
           .dropna()
           .astype(int)
           .clip(lower=1, upper=444))
    df = distance_df.copy()
    df["__pos__"] = pos.values

    # --- Segment definition (bottom track) ---
    # (label, start, end, type_group)
    raw_segments = [
        ("Extracellular_Domain", 1,   54),
        ("Helical_1",            55,  75),
        ("Cytoplasmic_1",        76,  88),
        ("Helical_2",            89,  110),
        ("Extracellular_2",      111, 127),
        ("Helical_3",            128, 150),
        ("Cytoplasmic_2",        151, 170),
        ("Helical_4",            171, 191),
        ("Extracellular_3",      192, 222),
        ("Helical_5",            223, 243),
        ("Cytoplasmic_3",        244, 304),
        ("Helical_6",            305, 326),
        ("Extracellular_4",      327, 342),
        ("Helical_7",            343, 366),
        ("Cytoplasmic_4",          367, 444),
    ]

    def segment_group(name: str) -> str:
        if name.startswith("Helical"):
            return "Helical"
        if name.startswith("Cytoplasmic"):
            return "Cytoplasmic"
        if name.startswith("Extracellular"):
            return "Extracellular"
        return "Other"

    # Consistent colors by group
    group_colors = {
        "Extracellular": "#3B82F6",  # blue
        "Helical":       "#F59E0B",  # amber
        "Cytoplasmic":   "#10B981",  # green
        "Other":         "#9CA3AF",  # gray
    }

    # --- Layout sizing ---
    docks = list(df["dock"].dropna().astype(str).unique())
    n_docks = len(docks)

    x_min, x_max = 1, 560
    segment_band_height = 0.5  # height of the bottom colored track
    row_height = 0.4           # per-dock row height
    gap = 0.15                 # gap between tracks
    top_extra = 0.8            # space above last row (for optional values line and title)

    total_height = segment_band_height + gap + n_docks * row_height + top_extra
    fig_height = max(3.0, total_height + 1.0)

    fig, ax = plt.subplots(figsize=(12, fig_height))

    # --- Bottom segment track ---
    y0 = 0.0
    for name, start, end in raw_segments:
        grp = segment_group(name)
        color = group_colors.get(grp, "#9CA3AF")
        # rectangle spans [start, end], adjust width so it includes end
        width = end - start + 1
        rect = Rectangle((start, y0), width, segment_band_height, facecolor=color, edgecolor="black", linewidth=0.5)
        ax.add_patch(rect)
        # Optional: annotate segment name (small)
        ax.text(start + width/2, y0 + segment_band_height/2, name,
                ha="center", va="center", fontsize=7, color="black")

    # Legend for groups (one per type)
    handles = []
    for grp, color in group_colors.items():
        # only include used groups
        if any(segment_group(n) == grp for n, *_ in raw_segments):
            handles.append(Rectangle((0, 0), 1, 1, facecolor=color, edgecolor="black", linewidth=0.5, label=grp))
    leg = ax.legend(handles=handles, loc="upper right", frameon=True, fontsize=8, title="Segment type")
    leg._legend_box.align = "left"

    # --- Rows for each dock ---
    # starting y for first dock row
    base_y = y0 + segment_band_height + gap
    tick_height = 0.28  # height of the small red vertical ticks

    for i, dock in enumerate(docks):
        row_y = base_y + i * row_height
        # label on the left
        ax.text(x_min - 5, row_y + tick_height/2, str(dock), ha="right", va="center", fontsize=9)

        # unique, sorted positions for this dock
        xs = (df.loc[df["dock"].astype(str) == dock, "__pos__"]
                .dropna()
                .astype(int)
                .clip(lower=x_min, upper=x_max)
                .unique())
        xs = np.sort(xs)

        if xs.size:
            # draw short vertical red ticks
            ax.vlines(xs, ymin=row_y, ymax=row_y + tick_height, colors="red", linewidth=1.0)

    # --- Optional: plot a dataset aligned to x=1..444 on top ---
    if values is not None:
        arr = np.asarray(values)
        if arr.shape[0] != (x_max - x_min + 1):
            raise ValueError(f"'values' must have length 444 (got {arr.shape[0]}).")
        vx = np.arange(x_min, x_max + 1)
        style = {"linewidth": 1.5}
        if values_style:
            style.update(values_style)
        top_y = base_y + n_docks * row_height + 0.2
        # Shift and scale the values to sit nicely in the top area (min->top_y, max->top_y+0.8)
        vmin, vmax = float(np.nanmin(arr)), float(np.nanmax(arr))
        if np.isfinite(vmin) and np.isfinite(vmax) and vmax > vmin:
            scaled = (arr - vmin) / (vmax - vmin)
        else:
            scaled = np.zeros_like(arr)
        scaled = top_y + 0.8 * scaled
        ax.plot(vx, scaled, **style)
        ax.text(x_min - 5, top_y + 0.4, "dataset", ha="right", va="center", fontsize=9, fontstyle="italic")

    # --- Axes cosmetics ---
    ax.set_xlim(x_min - 1, x_max + 1)
    ax.set_xlabel("Position")
    if title:
        ax.set_title(title, pad=12)

    # Hide y axis; keep a clean look
    ax.get_yaxis().set_visible(False)

    # Light grid to help read x positions
    ax.set_axisbelow(True)
    ax.grid(axis="x", linestyle=":", linewidth=0.6)

    # Add a left margin so dock labels are visible
    plt.subplots_adjust(left=0.08, right=0.98, top=0.92, bottom=0.08)
    return fig, ax


# -----------------------
# Example usage:
# -----------------------
# Suppose you already have a DataFrame `distance_df` with columns 'dock' and 'Distance'.
# Example (remove if you already have real data):
if __name__ == "__main__":
    # # Dummy example data
    # rng = np.random.default_rng(42)
    # example = []
    # for d in ["DockA", "DockB", "DockC"]:
    #     # random unique positions per dock
    #     positions = np.unique(rng.integers(1, 445, size=40))
    #     for p in positions:
    #         example.append({"dock": d, "Distance": int(p)})
    # distance_df = pd.DataFrame(example)

    # # Optional dataset values aligned to positions 1..444 (e.g., random)
    # values = np.sin(np.linspace(0, 12*np.pi, 444)) + 0.1 * rng.normal(size=444)

    fig, ax = plot_docks_with_segments(distance_df, values=values, title="Dock distances with domain segments")
    plt.show()


KeyError: 'atom1_residue_num'